In [1]:
# Stage 1: Download HLA types reference file
from pathlib import Path
import urllib.request

DATA_DIR = Path(".")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/HLA_types/20181129_HLA_types_full_1000_Genomes_Project_panel.txt

url = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/HLA_types/20181129_HLA_types_full_1000_Genomes_Project_panel.txt"
dest = DATA_DIR / Path(url).name

if dest.exists():
    print(f"[cached]   {dest.name} ({dest.stat().st_size:,} bytes)")
else:
    print(f"[download] {url}")
    urllib.request.urlretrieve(url, dest)
    print(f"           -> {dest} ({dest.stat().st_size:,} bytes)")


[cached]   20181129_HLA_types_full_1000_Genomes_Project_panel.txt (202,701 bytes)


In [2]:
# ===============================================================================================
# Stage 2: Read rows

EXPECTED_COLUMNS = [
    "Region", "Population", "Sample ID",
    "HLA-A 1", "HLA-A 2",
    "HLA-B 1", "HLA-B 2",
    "HLA-C 1", "HLA-C 2",
    "HLA-DQB1 1", "HLA-DQB1 2",
    "HLA-DRB1 1", "HLA-DRB1 2",
]

with open(dest) as f:
    header = f.readline().rstrip("\n").split("\t")
    header_cols_stripped = [h.strip() for h in header]
    if header_cols_stripped != EXPECTED_COLUMNS:
        raise ValueError(f"Unexpected header: {header_cols_stripped}")
    col = {name: idx for idx, name in enumerate(header_cols_stripped)}
    rows = []
    for line in f:
        if not line.strip():
            continue
        r = line.rstrip("\n").split("\t")
        r = [v.strip() for v in r]
        if len(r) != len(header):
            raise ValueError(f"Malformed row (got {len(r)} fields, expected {len(header)}): {r}")
        rows.append(r)

print(f"Rows: {len(rows)}")
print(f"First row: {rows[0]}")

Rows: 2693
First row: ['AFR', 'ACB', 'HG01879', '23:01', '68:02', '13:02', '42:01', '08:04', '17:01', '02:02', '04:02', '03:02', '09:01']


In [3]:
# ===============================================================================================
# Stage 3: Collect sample IDs

sample_ids = set()
for row in rows:
    sid = row[col["Sample ID"]]
    if sid in sample_ids:
        raise ValueError(f"Duplicate Sample ID: {sid}")
    sample_ids.add(sid)

print(f"Unique sample IDs: {len(sample_ids)}")

Unique sample IDs: 2693


In [4]:
# ===============================================================================================
# Stage 4: Load trios JSON and collect its sample IDs
 
import json
 
trios_path = DATA_DIR / "families_1000G_trios.json"
with open(trios_path) as f:
    trios_data = json.load(f)
 
trios_sample_ids = set()
for key, trio in trios_data["family_trios"].items():
    for role in ("child", "father", "mother"):
        sid = trio["members"][role]["sample_id"]
        trios_sample_ids.add(sid)
 
print(f"Trios sample IDs: {len(trios_sample_ids)}")

Trios sample IDs: 1793


In [5]:
# ===============================================================================================
# Stage 5: Find trios sample IDs missing from HLA sample IDs
 
missing_sample_ids = set()
for sid in trios_sample_ids:
    if sid not in sample_ids:
        missing_sample_ids.add(sid)
 
print(f"Missing from HLA: {len(missing_sample_ids)}")
print(f"Sample missing IDs: {sorted(missing_sample_ids)[:10]}")

Missing from HLA: 633
Sample missing IDs: ['HG00405', 'HG00408', 'HG00420', 'HG00423', 'HG00429', 'HG00438', 'HG00444', 'HG00447', 'HG00450', 'HG00453']


In [6]:
# ===============================================================================================
# Stage 6: Count trios where all three members have HLA typing
 
complete_trios_count = 0
for key, trio in trios_data["family_trios"].items():
    child_sid  = trio["members"]["child"]["sample_id"]
    father_sid = trio["members"]["father"]["sample_id"]
    mother_sid = trio["members"]["mother"]["sample_id"]
    if child_sid in sample_ids and father_sid in sample_ids and mother_sid in sample_ids:
        complete_trios_count += 1
 
print(f"Trios with HLA typing for all three members: {complete_trios_count}")

Trios with HLA typing for all three members: 7
